# Кейс 3. Шаг 1б: дозапросы разведки

Работает только с Parquet на Drive (собран ноутбуком `case3_00_load_recon`), ничего не скачивает. Время выполнения: несколько минут. Вывод сохраняется в `recon2_report.txt` рядом с Parquet.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/case3"
import os, json, numpy as np, pandas as pd, duckdb
from google.colab import drive
drive.mount("/content/drive")
PARQUET = f"{DRIVE_DIR}/parquet"
con = duckdb.connect()
con.execute("SET preserve_insertion_order = false")
for f in sorted(os.listdir(PARQUET)):
    if f.endswith(".parquet") and not f.startswith("user_connection_logs_"):
        con.execute(f"CREATE OR REPLACE VIEW {f[:-8]} AS SELECT * FROM '{PARQUET}/{f}'")
con.execute(f"CREATE OR REPLACE VIEW conn AS SELECT * FROM read_parquet('{PARQUET}/user_connection_logs_*.parquet')")

REPORT = []
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
def out(*a):
    s = " ".join(str(x) for x in a); print(s); REPORT.append(s)
def show(title, sql, n=80):
    out(f"\n## {title}")
    try:
        df = con.sql(sql).df(); out(df.head(n).to_string(index=False) if len(df) else "(пусто)"); return df
    except Exception as e:
        out(f"ОШИБКА: {e}")
def save():
    open(f"{DRIVE_DIR}/recon2_report.txt", "w", encoding="utf-8").write("\n".join(REPORT))

## 1. Часовой пояс полей без пояса

`sessions.created_at` хранится с поясом и переведён в UTC, это эталон. Для каждого поля без пояса ищем сдвиг (в часах), при котором его суточный профиль активности лучше всего совпадает с эталоном. Сдвиг 0 ⇒ поле в UTC, сдвиг +3 ⇒ поле в московском времени.

In [ ]:
SOURCES = {
    "sessions.created_at (эталон, UTC)": "SELECT hour(created_at) h, count(*) n FROM sessions GROUP BY 1",
    "users.created_at": "SELECT hour(created_at) h, count(*) n FROM users GROUP BY 1",
    "orders.created_at": "SELECT hour(created_at) h, count(*) n FROM orders GROUP BY 1",
    "transactions.created_at": "SELECT hour(created_at) h, count(*) n FROM transactions GROUP BY 1",
    "subscribe.created_at": "SELECT hour(created_at) h, count(*) n FROM subscribe GROUP BY 1",
    "conn.created_at": "SELECT hour(created_at) h, count(*) n FROM conn GROUP BY 1",
    "adv_view_history.created_at": "SELECT hour(created_at) h, count(*) n FROM adv_view_history GROUP BY 1",
    "proxy_user_connection_logs.created_at": "SELECT hour(created_at) h, count(*) n FROM proxy_user_connection_logs GROUP BY 1",
}
prof = {}
for name, sql in SOURCES.items():
    d = con.sql(sql).df().dropna().set_index("h")["n"].reindex(range(24), fill_value=0)
    prof[name] = d / d.sum()
ref = prof["sessions.created_at (эталон, UTC)"].values
out("\n# 1. Сдвиг поля без пояса относительно UTC (лучшее совпадение суточного профиля)")
for name, p in prof.items():
    corr = {s: np.corrcoef(np.roll(p.values, -s), ref)[0, 1] for s in range(-6, 7)}
    best = max(corr, key=corr.get)
    out(f"   {name}: сдвиг {best:+d} ч (корреляция {corr[best]:.3f}; при 0 ч: {corr[0]:.3f}; при +3 ч: {corr[3]:.3f}); час пика {int(p.values.argmax())}")
show("orders.created_at − users.created_at для первого заказа: доля отрицательных разниц (заказ раньше регистрации ⇒ пояса разные)", """
    WITH f AS (SELECT user_id, min(created_at) t FROM orders GROUP BY 1)
    SELECT round(avg((f.t < u.created_at)::INT), 4) share_order_before_reg,
           round(avg((f.t < u.created_at - INTERVAL 3 HOUR)::INT), 4) share_before_reg_minus_3h
    FROM f JOIN users u ON u.id = f.user_id""")
show("subscribe.created_at − orders.updated_at того же заказа, часы", """
    SELECT round(epoch(s.created_at - o.updated_at) / 3600)::BIGINT diff_h, count(*) n
    FROM subscribe s JOIN orders o ON o.id = s.order_id GROUP BY 1 ORDER BY n DESC LIMIT 5""")
save()

## 2. Запросы, упавшие в первой разведке (исправлены алиасы)

In [ ]:
out("\n# 2. Повтор упавших запросов")
show("логи подключений по месяцам", """
    SELECT strftime(created_at, '%Y-%m') m, count(*) n, count(DISTINCT user_id) users,
           round(avg((subscribe_id IS NULL)::INT), 3) no_sub_share, count(DISTINCT CAST(created_at AS DATE)) n_days
    FROM conn GROUP BY 1 ORDER BY 1""")
show("подключения по дням, февраль–май", """
    SELECT CAST(created_at AS DATE) d, count(*) n, count(DISTINCT user_id) users FROM conn
    WHERE created_at < '2026-06-01' GROUP BY 1 ORDER BY 1""", n=130)
show("топ-5 нод по подключениям в каждом месяце", """
    WITH c AS (SELECT strftime(created_at, '%Y-%m') m, node_id, count(*) n, avg((subscribe_id IS NULL)::INT) no_sub
               FROM conn GROUP BY 1, 2),
         r AS (SELECT *, row_number() OVER (PARTITION BY m ORDER BY n DESC) rk, n / sum(n) OVER (PARTITION BY m) share_n FROM c)
    SELECT r.m, r.rk, substr(r.node_id, 1, 8) node, round(r.share_n, 3) share_n, round(r.no_sub, 3) no_sub,
           nd.free_pool, nd.auto_select, l.country_code
    FROM r LEFT JOIN nodes nd ON nd.id = r.node_id LEFT JOIN locations l ON l.id = nd.location_id
    WHERE r.rk <= 5 ORDER BY r.m, r.rk""")
show("просмотры рекламы по месяцам", """
    SELECT strftime(created_at, '%Y-%m') m, count(*) n_views, count(DISTINCT user_id) users,
           round(count(*) / count(DISTINCT user_id), 2) per_user
    FROM adv_view_history GROUP BY 1 ORDER BY 1""")
show("mobile_version: платформы", "SELECT platform, count(*) n, min(created_at) first_at, max(created_at) last_at FROM mobile_version GROUP BY 1")
show("users.trial_used_at", "SELECT count(*) users, count(trial_used_at) with_trial, min(trial_used_at) first_at, max(trial_used_at) last_at FROM users")
show("orders: trial × status", """
    SELECT trial, status, count(*) n, min(created_at) first_at, max(created_at) last_at FROM orders GROUP BY 1, 2 ORDER BY 1, 2""")
show("длительность подписки по заказу с триалом, дни", """
    SELECT round(epoch(s.expired_time - s.created_at) / 86400, 1) n_days, count(*) n
    FROM subscribe s JOIN orders o ON o.id = s.order_id WHERE o.trial = 't' GROUP BY 1 ORDER BY n DESC LIMIT 10""")
show("длительность подписки по обычному оплаченному заказу, дни", """
    SELECT o.period_id, round(epoch(s.expired_time - s.created_at) / 86400) n_days, count(*) n
    FROM subscribe s JOIN orders o ON o.id = s.order_id WHERE o.trial = 'f' GROUP BY 1, 2 ORDER BY n DESC LIMIT 12""")
save()

## 3. Дополнительные проверки для витрины и гипотез

In [ ]:
out("\n# 3. Дополнительные проверки")
show("product_trial (одна строка)", "SELECT * FROM product_trial")
show("заказы по роли пользователя", """
    SELECT u.role, count(*) orders, count(*) FILTER (WHERE o.status = 'paid') paid,
           count(DISTINCT o.user_id) users FROM orders o JOIN users u ON u.id = o.user_id GROUP BY 1""")
show("кто подключается без подписки: роль, по месяцам", """
    SELECT strftime(c.created_at, '%Y-%m') m, u.role, count(DISTINCT c.user_id) users, count(*) n
    FROM conn c JOIN users u ON u.id = c.user_id WHERE c.subscribe_id IS NULL GROUP BY 1, 2 ORDER BY 1, 2""")
show("кто смотрит рекламу: роль", """
    SELECT u.role, count(DISTINCT a.user_id) users, count(*) n_views
    FROM adv_view_history a JOIN users u ON u.id = a.user_id GROUP BY 1""")
show("отпечаток карты: на скольких пользователях одна карта", """
    SELECT n_users, count(*) cards FROM (SELECT card_fp, count(DISTINCT user_id) n_users FROM saved_payment_methods
    WHERE card_fp IS NOT NULL GROUP BY 1) GROUP BY 1 ORDER BY 1""")
show("способы оплаты у пользователей с оплаченным триалом", """
    WITH t AS (SELECT DISTINCT user_id FROM orders WHERE trial = 't' AND status = 'paid')
    SELECT count(*) trial_users,
           count(*) FILTER (WHERE EXISTS (SELECT 1 FROM saved_payment_methods s WHERE s.user_id = t.user_id)) with_saved_method,
           count(*) FILTER (WHERE EXISTS (SELECT 1 FROM saved_payment_methods s WHERE s.user_id = t.user_id AND s.card_fp IS NOT NULL)) with_card_fp
    FROM t""")
show("pending-заказы по месяцам и провайдерам (всплеск в мае–июне)", """
    SELECT strftime(created_at, '%Y-%m') m, provider_id, status, count(*) n FROM orders
    WHERE created_at >= '2026-04-01' AND created_at < '2026-08-01' GROUP BY ALL ORDER BY 1, 2, 3""")
show("доля неуспеха по периоду подписки (обычные заказы)", """
    SELECT period_id, count(*) n, round(avg((status = 'failed')::INT), 3) failed, round(avg((status = 'paid')::INT), 3) paid
    FROM orders WHERE trial = 'f' GROUP BY 1 ORDER BY 1""")
show("pre_payment у оплаченных обычных заказов по месяцам", """
    SELECT strftime(created_at, '%Y-%m') m, count(*) n, round(avg((pre_payment = 't')::INT), 3) pre_payment_share
    FROM orders WHERE trial = 'f' AND status = 'paid' GROUP BY 1 ORDER BY 1""")
show("subscribe: next_product_id / pre_payment / disabled_at", """
    SELECT pre_payment, pre_payment_notification, (next_product_id IS NOT NULL) has_next, (disabled_at IS NOT NULL) disabled, count(*) n
    FROM subscribe GROUP BY ALL ORDER BY n DESC""")
save()
out(f"\nОтчёт: {DRIVE_DIR}/recon2_report.txt")